<a href="https://www.kaggle.com/code/srikanthmachiraju/raft-finetuning-slm?scriptVersionId=316451463" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# RAFT Fine-Tuning 

## Overview

**RAFT (Retrieval-Augmented Fine-Tuning)** is a supervised fine-tuning strategy that trains a language model to answer questions from a retrieved context that contains both the relevant *oracle* passage and several *distractor* documents. This teaches the model to identify and reason from the correct source rather than relying on parametric memory — directly improving RAG pipeline performance at inference time without any architecture changes.

**Reference:** *RAFT: Adapting Language Model to Domain Specific RAG* (Zhang et al., 2024).

---

### End-to-End Pipeline

```
PDF documents
    └─► pdf_to_chunks.py     (GPT-4o vision → per-page .txt files)
    └─► raft_datagen.py      (GPT-4o → Q/A/D triplets → JSONL)
    └─► this notebook        (Unsloth + LoRA → fine-tuned Llama-3.2-1B)
```

### Dataset Schema

Each JSONL record contains:

| Field | Description |
|-------|-------------|
| `question` | Synthetic question generated from a document chunk |
| `context` | Dict with `sentences` — oracle + `num_distract` distractor passages (shuffled) |
| `oracle_context` | The single chunk that actually answers the question |
| `instruction` | Full `<DOCUMENT>…</DOCUMENT>` prompt fed to the model |
| `cot_answer` | Chain-of-thought answer with `##begin_quote##` citations and final `<ANSWER>:` tag |

### Key Training Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `max_seq_length` | 2048 | Covers instruction + 4 docs + CoT answer |
| `load_in_4bit` | True | NF4 quantisation — ~4× memory reduction |
| LoRA rank `r` | 16 | Balances expressivity vs. parameter count |
| `lora_alpha` | 16 | Effective scale = alpha/r = 1.0 |
| `learning_rate` | 2e-5 | Conservative for an instruction-tuned base |
| `gradient_accumulation_steps` | 8 | Effective batch size = 2 × 8 = 16 |
| `lr_scheduler_type` | cosine | Smooth decay; standard for short runs |

---

## Setup: Install Dependencies

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: 
    import subprocess; 
    is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: 
    is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install loguru sqlglot sqlparse swifter

## Step 1: Load the RAFT Dataset

Load the pre-generated RAFT training and evaluation splits from disk. Each split is a JSONL file where every record is a Q/A/D triplet.

- **`train.jsonl`** — 80% of generated examples, used for gradient updates  
- **`test.jsonl`** — held-out 20%, used for evaluation loss during training  

`tiktoken` is imported for optional token-count analysis of the dataset before training.

In [ ]:
import os
import tiktoken
import json
import pandas as pd

MAX_SEQ_LEN = 2048 

training_data_dir = '/kaggle/input/datasets/srikanthmachiraju/raft-finetuning-dataset/filtered/'
train_df = pd.read_json(os.path.join(training_data_dir, "train.jsonl"), lines=True)
test_df = pd.read_json(os.path.join(training_data_dir, "test.jsonl"), lines=True)
print(f"Number of samples: train={len(train_df)}, test={len(test_df)}")
train_df.head()

In [ ]:
train_df[train_df["type"]=="distractor"].head()

## Step 2: Convert to HuggingFace `Dataset` Format

Convert the pandas DataFrames into HuggingFace `Dataset` objects. This enables:
- Efficient batched `.map()` transformations in subsequent steps
- Compatibility with `SFTTrainer` which expects HF datasets
- Arrow-backed memory mapping for large datasets

In [ ]:
from datasets import Dataset

# Convert your pandas DataFrames to HF Datasets
train_dataset = Dataset.from_pandas(train_df)
eval_dataset  = Dataset.from_pandas(test_df)

# Check the structure
print(train_dataset)
print(train_dataset.column_names)
# print(train_dataset[0])   # view one example

## Step 3: Load Base Model and Attach LoRA Adapters

Load **Llama-3.2-1B-Instruct** using [Unsloth](https://github.com/unslothai/unsloth)'s optimised `FastLanguageModel`, then attach **LoRA (Low-Rank Adaptation)** adapters to the attention and MLP projection layers.

### Why LoRA?
Full fine-tuning of a 1B parameter model requires updating all weights (~4 GB in fp16). LoRA instead injects small trainable rank decomposition matrices (`r=16`) into each target layer. This reduces trainable parameters by **~99%** while preserving most of the base model's capabilities.

### Target modules
All six projection matrices are adapted — `q/k/v/o_proj` (attention) and `gate/up/down_proj` (MLP) — which gives the model maximum flexibility to learn the RAFT task format.

### Memory optimisation
- **4-bit NF4 quantisation** reduces the frozen base weights to ~500 MB GPU memory  
- **`use_gradient_checkpointing="unsloth"`** trades recomputation for ~30% less VRAM during the backward pass

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

lora_rank = 8 # reduced to 8 after noticing overfitting

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, 
    dtype=None,
)

# Recommended for Llama-3.2
tokenizer = get_chat_template(
    tokenizer, 
    chat_template="llama-3.2"   # or "llama-3.2" if available in your Unsloth version
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 2 * lora_rank,
    lora_dropout = 0, # set to 0 for optimized training
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 2025,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## Step 4: Apply Chat Template Formatting

Transform each raw Q/A/D record into a single text string using the **Llama-3.2 chat template**. This step is critical because the model was instruction-tuned with a specific prompt structure; maintaining that structure during fine-tuning prevents catastrophic forgetting of the base chat format.

### Prompt structure per example

```
<|system|>  You are a helpful assistant…
<|user|>              
            ### Context: <DOCUMENT>…</DOCUMENT> prompt
<|assistant|> <chain-of-thought reasoning> with ##begin_quote## and ##end_quote##
             <ANSWER>: …
```

### Why include distractors in the prompt?
RAFT deliberately exposes the model to noisy context during training. With probability `p=0.8` the oracle is present; with `p=0.2` it is replaced. This teaches robustness — the model learns to either find the answer or correctly abstain.

### Column cleanup
`remove_columns=train_dataset.column_names` drops all original columns, leaving only `text`. This avoids padding mismatches in the collator when column schemas differ between examples.

In [ ]:
_SYSTEM_PROMPT = "You are a helpful assistant that answers questions using the provided context ONLY. \
DO NOT use any information that is not included in the <Retrieved Documents>.  \
You should Answer ### Question STRICTLY in this FORMAT: \
### Step-by-step reasoning: Use several quotes from <Retrieved Documents>: \
##begin_quote## [Relevant text 1] ##end_quote## \
##begin_quote## [Relevant text 2] ##end_quote## \
Then think step-by-step. <ANSWER>Answer here...</ANSWER>" 

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    texts = []
    for qn, ctx, oracle, instr, ans, _type in zip(
        examples["question"],
        examples["context"],
        examples["oracle_context"],
        examples["instruction"],
        examples["cot_answer"],
        examples["type"]
    ):
        
        messages = [
            {"role": "system", "content": _SYSTEM_PROMPT },
            {"role": "user", "content": f"<Retrieved Documents>: \n{instr}"}, # Context + Question
            {"role": "assistant", "content": f"{ans}"}   # COT style answer
        ]
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        texts.append(text) # + tokenizer.eos_token)
    
    return {"text": texts}

# Apply formatting
train_ds = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names   # Clean old columns, keep only "text"
)

eval_ds = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=eval_dataset.column_names
)

# Verify
train_ds[0]

## Step 5: Cache Formatted Datasets to Disk

Persist the formatted datasets in HuggingFace Arrow format. This is useful when:
- Iterating on training hyperparameters without re-running the expensive formatting step
- Re-using the exact same formatted data across kernel restarts

The next cell shows how to reload from disk, skipping Steps 1–4 entirely.

In [ ]:
# After creating train_ds and eval_ds, save them
train_ds.save_to_disk("raft_train_hf")
eval_ds.save_to_disk("raft_eval_hf")

In [ ]:
# Later, load them directly (no need to convert again)
from datasets import load_from_disk
train_ds = load_from_disk("raft_train_hf")
eval_ds = load_from_disk("raft_eval_hf")

## Step 6: Configure and Initialise the SFT Trainer

Configure `trl.SFTTrainer` with `transformers.TrainingArguments`. Key decisions:

| Argument | Value | Notes |
|----------|-------|-------|
| `per_device_train_batch_size` | 2 | Constrained by GPU VRAM with 4-bit model |
| `gradient_accumulation_steps` | 8 | Effective batch = 16; smooths gradient noise |
| `num_train_epochs` | 1 | Single pass — RAFT data quality > quantity |
| `learning_rate` | 2e-5 | Below the typical 5e-5 to protect instruction-following |
| `fp16` | True | Mixed-precision training; compatible with T4/A10 |
| `eval_strategy` | steps (every 5) | Frequent eval to detect divergence early |
| `optim` | adamw_torch | Decoupled weight decay; preferred over legacy adamw |
| `lr_scheduler_type` | cosine | Smooth warmup-free decay for short runs |
| `save_strategy` | no | Model is saved explicitly in Step 10 after merging |

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

training_args = TrainingArguments(
    output_dir="llama32_1bn_instruct_raft", #This will also be used as your huggingfacehub model id name
    report_to="none", #Leave this to be blank if you don't want to use wandb
    per_device_train_batch_size=2,    # small batches if quantized
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    warmup_steps = 10,
    max_steps = 120,
    learning_rate=2e-5,
    save_strategy="no",
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=10,
    logging_steps=1,
    seed=42,
    optim="adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds, 
    args=training_args,
    dataset_text_field="text",
    packing = False, # Can make training 5x faster for short sequences.
)   

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

## Step 7: GPU Memory Snapshot (Pre-Training)

Record baseline GPU memory stats before training begins. These values are used post-training to calculate peak memory consumption and validate that the configuration fits within the available VRAM budget. If `start_gpu_memory / max_memory > 0.8`, consider reducing `per_device_train_batch_size` or enabling `load_in_8bit`.

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

## Step 8: Train the Model

Launch the training loop. `trainer.train()` returns a `TrainOutput` object containing:
- `global_step` — total number of optimiser steps taken  
- `training_loss` — average loss over the run  
- `metrics` — timing and throughput statistics  

Training progress and eval loss are logged every 5 steps. Monitor for:
- **Decreasing train loss** — confirms the model is learning the RAFT format  
- **Eval loss tracking train loss** — large divergence indicates overfitting; consider reducing epochs or increasing dropout

In [ ]:
trainer_stats = trainer.train()

## Step 9: Review Training Statistics

Inspect the `TrainOutput` object. Key metrics to review:

- **`train_loss`** — final training loss; should be below 1.0 for a well-fitted RAFT model  
- **`train_runtime`** — total wall-clock training time in seconds  
- **`train_samples_per_second`** — throughput; use this to extrapolate cost for longer runs  
- **`train_steps_per_second`** — compare against expected steps given your batch config

In [ ]:
trainer_stats

Verify GPU usage after training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
import gc; gc.collect()

## Step 10: Save Merged Model in 16-bit

Merge the LoRA adapter weights back into the base model and save as a single 16-bit checkpoint. This produces a standard HuggingFace model directory that can be:
- Loaded with `AutoModelForCausalLM.from_pretrained()` — no LoRA dependency required
- Quantised further (GGUF, GPTQ) for local deployment
- Pushed directly to the Hub

`save_method="merged_16bit"` is preferred over `lora_only` for portability and over `merged_4bit` when upload size is not a constraint.

In [ ]:
model.save_pretrained_merged(
    save_directory = "llama32_1bn_instruct_raft",     
    tokenizer = tokenizer,
    save_method = "merged_16bit",        
)

## Step 11: Authenticate with Hugging Face Hub

Retrieve the HF API token from Kaggle Secrets and authenticate. This scopes authentication to the current session without persisting the token to disk. The token requires **write** access to the target repository.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

## Step 12: Push Model and Tokenizer to Hugging Face Hub

Upload the merged model weights and tokenizer to the specified Hub repository. Both must be pushed — the tokenizer carries the chat template configuration which is required for correct inference.

In [ ]:
hf_model_path = "sriksmachi/llama32_1bn_instruct_raft"

# Use model.push_to_hub() to upload
model.push_to_hub(hf_model_path)

# Don't forget to push the tokenizer as well
tokenizer.push_to_hub(hf_model_path)

## Step 13: Pull Model from HF and generate answers

Pulls the model and tokenizer from HF, and generate answers. It is important to enable the inference mode

In [ ]:
from unsloth import FastLanguageModel

fn_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=hf_model_path,
    max_seq_length=MAX_SEQ_LEN,
)

FastLanguageModel.for_inference(fn_model)

In [ ]:
from transformers import TextStreamer
from tqdm import tqdm

validation_df = pd.read_json(os.path.join(training_data_dir, "validation.jsonl"), lines=True)

baseline_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = MAX_SEQ_LEN, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
)

FastLanguageModel.for_inference(baseline_model)

## Step 14: Model Evaluation for oracle and distractor samples

In [ ]:
MAX_NEW_TOKENS = 512

def generate_answer(instruction, model):
    """Generate an answer using a HuggingFace text-generation pipeline."""
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": f"<Retrieved Documents>: {instruction}"}, # Context + Question
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens = MAX_NEW_TOKENS, use_cache = False, temperature = 0.1, min_p = 0.1,do_sample = False)
    input_length = inputs.shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
    return generated_text

# Response for oracle docs
oracle_sample = validation_df[validation_df["type"] == "oracle"].iloc[1]
print(f"====================Question===================\n")
print(f"{oracle_sample["question"]}")
print(f"===================INSTRUCTION================\n")
print(f"{oracle_sample["instruction"]}")
print(f"=====================Baseline Model Response===================\n")
print(generate_answer(oracle_sample["instruction"], baseline_model))


In [ ]:
%%time
print(f"=====================Finetuned Model Response===================\n")
print(generate_answer(oracle_sample["instruction"], fn_model))

### Distractor Sample

In [ ]:
# Response for oracle docs
for i in range(3):
    distractor_sample = validation_df[validation_df["type"] == "distractor"].iloc[i]
    print(f"===========================SAMPLE {i}======================\n")
    print(f"===================INSTRUCTION================\n")
    print(f"{distractor_sample["instruction"]}")
    print(f"=====================Baseline Model Response===================\n")
    print(generate_answer(distractor_sample["instruction"], baseline_model))
    print(f"=====================Finetuned Model Response===================\n")
    print(generate_answer(distractor_sample["instruction"], fn_model))
    print(f"===========================================================\n\n")